# 01 — Baseline

Treina a CNN de referência. Todas as ablações do notebook 02 partem exatamente desta configuração e mudam um único argumento.

In [ ]:
# Setup: no Colab (site ou extensão do VS Code) clona o repositório do GitHub
# e instala as dependências; localmente só ajusta o sys.path.
# USE_DRIVE guarda pesos, JSONs e figuras no Google Drive para sobreviverem ao
# fim da sessão (o notebook 04 precisa dos pesos).
import os, sys
USE_DRIVE = True
# Detecta o Colab pela pasta /content em vez de "google.colab": no kernel da
# extensão do VS Code esse módulo pode não estar carregado.
IN_COLAB = os.path.isdir("/content") or "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/deep-vision-classification"
    # O código que roda é o do GitHub, não o da sua máquina: faça push antes.
    if not os.path.exists(REPO):
        !git clone -q https://github.com/davimoljo/deep-vision-classification.git {REPO}
    else:
        !git -C {REPO} pull -q
    %cd {REPO}
    !pip install -q -r requirements.txt
    if USE_DRIVE:
        if not os.path.isdir("/content/drive/MyDrive"):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print("Não montou o Drive:", e)
                print("No VS Code use 'Colab: Mount Google Drive to Server...' e rode esta célula de novo.")
        if os.path.isdir("/content/drive/MyDrive"):
            os.environ["OUTPUT_ROOT"] = "/content/drive/MyDrive/deep-vision-classification"
            print("Saídas em", os.environ["OUTPUT_ROOT"])
        else:
            print("ATENÇÃO: sem Drive, resultados e pesos somem quando o servidor desligar.")
    ROOT = REPO
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

In [ ]:
from src.pipeline import (DEFAULTS, STRATEGIES, build_cnn, build_from_config,
                          build_pretrained, count_parameters, evaluate_on_test, fit,
                          get_loaders, list_results, load_best_weights, load_result,
                          result_exists)
from src.plots import comparison_table, per_class_accuracy, plot_confusion, plot_curves
DEFAULTS

## Configuração

`BASELINE = {}` significa os valores padrão de `BaselineCNN` (3 blocos, 32 filtros, kernel 3, 2 convs/bloco, BN, dropout 0.3, FC 256, flatten).

In [ ]:
RUN = "baseline"
BASELINE = {}
loaders = get_loaders(DEFAULTS["batch_size"], img_size=32, augment=True)
model = build_cnn(**BASELINE)
total, trainable = count_parameters(model)
print(model)
print(f"{total:,} parâmetros ({trainable:,} treináveis)")

## Treino

In [ ]:
result = fit(model, loaders, RUN)

## Curvas (diagnóstico de overfitting)

In [ ]:
plot_curves(RUN);